In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv.zip
/kaggle/input/description.md
/kaggle/input/train.csv
/kaggle/input/test.csv.zip
/kaggle/input/tabular-playground-series-dec-2021/sample_submission.csv.zip
/kaggle/input/tabular-playground-series-dec-2021/test.csv
/kaggle/input/tabular-playground-series-dec-2021/sample_submission.csv
/kaggle/input/tabular-playground-series-dec-2021/train.csv.zip
/kaggle/input/tabular-playground-series-dec-2021/description.md
/kaggle/input/tabular-playground-series-dec-2021/train.csv
/kaggle/input/tabular-playground-series-dec-2021/test.csv.zip


In [2]:
df_train = pd.read_csv("../input/tabular-playground-series-dec-2021/train.csv")
df_test = pd.read_csv("../input/tabular-playground-series-dec-2021/test.csv")




In [3]:
def reduce_mem_usage(df, verbose=True):
    numerics = ["int16", "int32", "int64", "float16", "float32", "float64"]
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(
            "Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)".format(
                end_mem, 100 * (start_mem - end_mem) / start_mem
            )
        )
    return df




In [4]:
df_train = reduce_mem_usage(df_train)
df_test = reduce_mem_usage(df_test)



Mem. usage decreased to 233.46 Mb (84.8% reduction)
Mem. usage decreased to 25.56 Mb (84.8% reduction)


In [5]:
df_train.head(5)



,Id,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
0,3440475,2634,132,3,166,38,1247,251,210,119,...,0,0,0,0,0,0,0,0,0,2
1,2470812,2769,89,5,633,10,626,177,209,178,...,0,0,0,0,0,0,0,0,0,2
2,536780,2749,46,30,127,239,2569,206,216,144,...,0,0,0,0,0,0,0,0,0,2
3,3115135,2574,155,20,279,1,1932,205,189,199,...,0,0,0,0,0,0,0,0,0,2
4,81861,2779,91,19,523,-2,2976,240,246,105,...,0,0,0,0,0,0,0,0,0,2


In [6]:
# Fix: make this a valid Python statement (was causing a syntax error if executed).
print("nan values in dataframe?")
df_train.isna().sum()



nan values in dataframe?


Id                                    0
Elevation                             0
Aspect                                0
Slope                                 0
Horizontal_Distance_To_Hydrology      0
Vertical_Distance_To_Hydrology        0
Horizontal_Distance_To_Roadways       0
Hillshade_9am                         0
Hillshade_Noon                        0
Hillshade_3pm                         0
Horizontal_Distance_To_Fire_Points    0
Wilderness_Area1                      0
Wilderness_Area2                      0
Wilderness_Area3                      0
Wilderness_Area4                      0
Soil_Type1                            0
Soil_Type2                            0
Soil_Type3                            0
Soil_Type4                            0
Soil_Type5                            0
Soil_Type6                            0
Soil_Type7                            0
Soil_Type8                            0
Soil_Type9                            0
Soil_Type10                           0


In [7]:
df_train.dtypes



Id                                    int32
Elevation                             int16
Aspect                                int16
Slope                                  int8
Horizontal_Distance_To_Hydrology      int16
Vertical_Distance_To_Hydrology        int16
Horizontal_Distance_To_Roadways       int16
Hillshade_9am                         int16
Hillshade_Noon                        int16
Hillshade_3pm                         int16
Horizontal_Distance_To_Fire_Points    int16
Wilderness_Area1                       int8
Wilderness_Area2                       int8
Wilderness_Area3                       int8
Wilderness_Area4                       int8
Soil_Type1                             int8
Soil_Type2                             int8
Soil_Type3                             int8
Soil_Type4                             int8
Soil_Type5                             int8
Soil_Type6                             int8
Soil_Type7                             int8
Soil_Type8                      

In [8]:
df_train.describe()




,Id,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
count,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,...,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06,3.600000e+06
mean,2.000069e+06,2.980147e+03,1.515727e+02,1.510050e+01,2.712761e+02,5.166790e+01,1.766483e+03,2.118390e+02,2.210576e+02,1.408136e+02,...,3.746250e-02,3.785972e-02,1.201639e-02,1.604194e-02,1.068556e-02,1.219861e-02,4.074917e-02,3.924222e-02,3.158944e-02,1.771544e+00
std,1.154820e+06,2.890440e+02,1.099497e+02,8.546351e+00,2.265100e+02,6.822503e+01,1.315661e+03,3.075498e+01,2.223373e+01,4.367894e+01,...,1.898923e-01,1.908569e-01,1.089587e-01,1.256368e-01,1.028172e-01,1.097716e-01,1.977086e-01,1.941708e-01,1.749044e-01,8.940261e-01
min,0.000000e+00,1.773000e+03,-3.300000e+01,-3.000000e+00,-8.200000e+01,-3.170000e+02,-2.870000e+02,-4.000000e+00,4.900000e+01,-5.300000e+01,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,9.997378e+05,2.760000e+03,6.000000e+01,9.000000e+00,1.100000e+02,4.000000e+00,8.220000e+02,1.980000e+02,2.100000e+02,1.150000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
50%,2.000172e+06,2.966000e+03,1.230000e+02,1.400000e+01,2.130000e+02,3.100000e+01,1.436000e+03,2.180000e+02,2.240000e+02,1.420000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
75%,3.000448e+06,3.217000e+03,2.470000e+02,2.000000e+01,3.610000e+02,7.800000e+01,2.365000e+03,2.330000e+02,2.370000e+02,1.690000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
max,3.999999e+06,4.383000e+03,4.050000e+02,6.400000e+01,1.602000e+03,6.320000e+02,7.666000e+03,2.970000e+02,2.790000e+02,2.720000e+02,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,7.000000e+00


In [9]:
def is_categorical(data, column):
    if len(data[column].unique()) <= 15:
        print(str(column) + ": " + str(data[column].unique()))
    return None




In [10]:
columns = df_train.columns.to_list()
for column in columns:
    is_categorical(df_train, column)



Wilderness_Area1: [0 1]
Wilderness_Area2: [0 1]
Wilderness_Area3: [1 0]
Wilderness_Area4: [0 1]
Soil_Type1: [0 1]
Soil_Type2: [0 1]
Soil_Type3: [0 1]
Soil_Type4: [0 1]
Soil_Type5: [0 1]
Soil_Type6: [0 1]
Soil_Type7: [0]
Soil_Type8: [0 1]
Soil_Type9: [0 1]
Soil_Type10: [0 1]
Soil_Type11: [0 1]
Soil_Type12: [0 1]
Soil_Type13: [0 1]
Soil_Type14: [0 1]
Soil_Type15: [0]
Soil_Type16: [0 1]
Soil_Type17: [0 1]
Soil_Type18: [0 1]
Soil_Type19: [0 1]
Soil_Type20: [0 1]
Soil_Type21: [0 1]
Soil_Type22: [0 1]
Soil_Type23: [1 0]
Soil_Type24: [0 1]
Soil_Type25: [0 1]
Soil_Type26: [0 1]
Soil_Type27: [0 1]
Soil_Type28: [0 1]
Soil_Type29: [0 1]
Soil_Type30: [0 1]
Soil_Type31: [0 1]
Soil_Type32: [0 1]
Soil_Type33: [0 1]
Soil_Type34: [0 1]
Soil_Type35: [0 1]
Soil_Type36: [0 1]
Soil_Type37: [0 1]
Soil_Type38: [0 1]
Soil_Type39: [0 1]
Soil_Type40: [0 1]
Cover_Type: [2 1 7 3 6 4 5]


In [11]:
df_train = df_train.drop(["Soil_Type7", "Soil_Type15"], axis=1)
df_test = df_test.drop(["Soil_Type7", "Soil_Type15"], axis=1)



In [12]:
targets = df_train.Cover_Type
df_train = df_train.drop(["Cover_Type"], axis=1)



In [13]:
targets.value_counts()



Cover_Type
2    2036254
1    1320866
3     176184
7      56125
6      10237
4        333
5          1
Name: count, dtype: int64

In [14]:
from sklearn import preprocessing

scaler = preprocessing.MinMaxScaler()

numeric_features = df_train.columns[1:11].to_list()

df_train[numeric_features] = scaler.fit_transform(df_train[numeric_features])
df_test[numeric_features] = scaler.transform(df_test[numeric_features])



In [15]:
train_X, val_X, train_y, val_y = train_test_split(df_train, targets, random_state=1)



In [16]:
# Fix: XGBoost expects class labels to be 0..num_class-1 by default.
# Encode y from {1..7} -> {0..6} for fitting, then decode back for submission.
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

train_y_enc = train_y.astype(np.int32) - 1
val_y_enc = val_y.astype(np.int32) - 1

# Fix: use CPU-safe tree_method to avoid GPU availability issues in Kaggle runtime.
model = XGBClassifier(
    tree_method="hist",
    learning_rate=0.3,
    random_state=1,
    n_estimators=300,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    num_class=7,
    eval_metric="mlogloss",
)

model.fit(train_X, train_y_enc, verbose=False)
pred_enc = model.predict(val_X)

acc = accuracy_score(val_y_enc, pred_enc)
print(f"accuracy_score:{acc}")



accuracy_score:0.9607444444444444


In [17]:
# Create submission with correct Id alignment and decode predictions back to {1..7}.
sub = pd.read_csv("../input/tabular-playground-series-dec-2021/sample_submission.csv")
test_pred_enc = model.predict(df_test)
sub["Cover_Type"] = test_pred_enc.astype(np.int32) + 1
sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub.shape)
print(sub.head())

Wrote submission.csv with shape: (400000, 2)
        Id  Cover_Type
0   814683           1
1  1357371           2
2  2106112           1
3  3483684           1
4  3960754           7
